# Week 1 Day 4 Lab - Open-Source, Local, Token Cost (free version)

## Open-weight models on UnderCover's claims

We send 20 of the claims in the queue to free models. `openrouter/free` picks a free model on OpenRouter for each request, and three open-weight models run on your own computer with Ollama. Every model gets a smaller prompt than on Day 2. Then we put accuracy, speed and memory side by side.

## 4 parts:

PART 1: The data and a smaller prompt

PART 2: Free models on OpenRouter

PART 3: Three models on your computer with Ollama

PART 4: Accuracy, speed and memory side by side

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This is the free version of the lab. It uses <code>openrouter/free</code>, so it needs no credit, and three models that run on your own computer. The full version with six paid models is <a href="week1day4.ipynb">week1day4.ipynb</a>. If your computer has no GPU, the CPU version in <a href="week1day4cpu.ipynb">week1day4cpu.ipynb</a> has local models small enough to run on the CPU.<br/><br/>
            OpenRouter's free models allow 50 requests a day on an account without credit, and this lab makes about 22 of them, a few more when a busy model means asking again. The local models have no limit.<br/><br/>
            Part 3 needs Ollama and about 8 GB of disk for the two small models, with 10 GB more for the optional third.
            </span>
        </td>
    </tr>
</table>

In [ ]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import ollama
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from openai import APIError, LengthFinishReasonError, OpenAI
from pydantic import BaseModel, Field
from tqdm.auto import tqdm

## Connect to OpenRouter

The OpenRouter key comes from the `.env` file you made on day 1 in the `undercover` folder.

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

## Part 1: The data and a smaller prompt

The seed data is in the repo's `data` folder, as on Day 2. Today we leave out the 5 manual claims. They depend on the claims manual or the customer's history, which the agent gets on Day 5, so the policy wording alone can't settle them. That leaves 20 claims. Some models take several minutes on all 20, so set `TINY = True` for a quicker run on 4 of them.

In [ ]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [ ]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
claims = [claim for claim in claims if labels[claim["claim_id"]]["difficulty"] != "manual"]
policies = {policy["policy_id"]: policy for policy in load("policies.json")}

print(f"{len(claims)} claims")

In [ ]:
decisions = pd.DataFrame([labels[claim["claim_id"]] for claim in claims])
decisions["decision"].value_counts()

12 of the 20 are approvals, so a model that approves every claim would get 60% of the decisions right. Keep that number in mind when we look at the results.

### A smaller prompt

On Day 2 the prompt had the whole policy wording, and the policy record and the claim as indented JSON. Today we trim it in three ways:

1. A short version of the policy wording, in `data/policy_wording_short.md`
2. Only the sections of it that matter for the type of claim
3. The policy record and the claim as compact JSON, without the ids and without the text of the documents

A smaller prompt costs less, runs faster and suits a small model on your own computer. It can also leave out something the model needs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: tokens</h2>
            <span style="color:#118AB2;">Models read and write tokens, which are pieces of words. In English a token is about three quarters of a word.<br/><br/>
            Providers charge per million tokens, with one price for the tokens you send and another for the tokens the model writes. The tokens it writes usually cost several times more, and reasoning tokens count as written.
            </span>
        </td>
    </tr>
</table>

In [ ]:
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")
short_wording = (DATA / "policy_wording_short.md").read_text(encoding="utf-8")

print(f"The policy wording is {len(policy_wording.split()):,} words")
print(f"The short version is {len(short_wording.split()):,} words")

The short wording has a section for each benefit, with headings that start with `## `. We split it into sections so we can pick the ones each claim needs.

In [ ]:
def split_sections(wording):
    """Split the wording into its opening lines and a dict of its sections, keyed by heading."""
    parts = wording.split("\n## ")
    sections = {part.split("\n")[0].strip(): "## " + part for part in parts[1:]}
    return parts[0], sections


opening, sections = split_sections(short_wording)
list(sections)

In [ ]:
BENEFITS = {
    "trip_cancellation": "Cancellation and Interruption",
    "trip_interruption": "Cancellation and Interruption",
    "trip_delay": "Trip Delay",
    "baggage_delay": "Baggage Delay",
    "baggage": "Baggage",
    "passport": "Passport",
    "emergency_medical": "Emergency Medical",
}


def make_wording(claim):
    """The sections of the short wording that this claim needs: the limits, its benefit, the add-ons if the policy has any, and the rules for every claim."""
    headings = ["Limits", BENEFITS[claim["claim_type"]]]
    if policies[claim["policy_id"]]["add_ons"]:
        headings.append("Add-ons")
    headings += ["Never covered", "Payment", "Claims"]
    return "\n\n".join([opening.strip()] + [sections[heading].strip() for heading in headings])

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using the Payment section.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""


def make_user_prompt(claim):
    """The policy record and the claim as compact JSON, without the ids and the text of the documents."""
    policy = policies[claim["policy_id"]]
    record = {key: value for key, value in policy.items() if key not in ["policy_id", "customer_id", "sold_through", "status"]}
    details = {key: value for key, value in claim.items() if key not in ["policy_id", "customer_id", "status"]}
    details["documents"] = [{"name": document["name"], "type": document["type"]} for document in claim["documents"]]
    return f"""# Policy record

{json.dumps(record)}

# Claim

{json.dumps(details)}
"""


def make_messages(claim):
    """Build the messages for one claim: the task and the wording it needs, then the policy record and the claim."""
    return [
        {"role": "system", "content": f"{INSTRUCTIONS}\n{make_wording(claim)}\n"},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

Here's the whole prompt for the first claim, exactly as the model sees it.

In [ ]:
messages = make_messages(claims[0])
print(messages[0]["content"])
print(messages[1]["content"])

## Part 2: Free models on OpenRouter

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Free models on OpenRouter</h2>
            <span style="color:#118AB2;">Free models on OpenRouter have an id that ends in <code>:free</code>, and most of them are open-weight. <code>openrouter/free</code> picks one of them for each request and moves on to another when one is busy. Our results record which model answered.<br/><br/>
            With <code>require_parameters</code> OpenRouter only picks models that support every parameter we send, including Structured Outputs. We leave the reasoning level at the model's default, since setting it made free models hang on Day 2.
            </span>
        </td>
    </tr>
</table>

We use the same Pydantic class as on Day 2 for Structured Outputs. The rationale comes before the decision so the model works through the claim before it decides.

In [ ]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

### Calling the LLM

`decide` is Day 2's function with two more things in the result: the model that answered and the number of input tokens.

Free models are busy at times. A request can come back with a rate limit error, or with an error in place of an answer, which the OpenAI SDK reports as a `TypeError`. Now and then a reply comes back empty.

The free models reason before they answer, and a reply usually takes 1,000 to 3,000 tokens. Once in a while a small one keeps on writing. `max_tokens=8000` cuts it off, and the OpenAI SDK raises a `LengthFinishReasonError`.

In each case `ask` tries again, up to three times, and `openrouter/free` picks another free model when it does.

In [ ]:
def ask(model, messages):
    """Send the messages to a free model on OpenRouter with Structured Outputs, and ask again if there's no answer."""
    for attempt in range(1, 4):
        try:
            response = openrouter.chat.completions.parse(
                model=model,
                messages=messages,
                response_format=ClaimDecision,
                max_tokens=8000,
                extra_body={"provider": {"require_parameters": True}},
            )
            if response.choices[0].message.parsed:
                return response
            print(f"An empty reply on try {attempt}, so we ask again")
        except (APIError, LengthFinishReasonError, TypeError) as error:
            print(f"{type(error).__name__} on try {attempt}, so we ask again")
    raise RuntimeError(f"No answer from {model} after 3 tries")


def decide(claim, model):
    """Ask a free model on OpenRouter to decide one claim."""
    start = time.time()
    response = ask(model, make_messages(claim))
    answer = response.choices[0].message.parsed
    return {
        "model": model,
        "where": "OpenRouter",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "answered_by": response.model,
        "input_tokens": response.usage.prompt_tokens,
        "seconds": round(time.time() - start, 1),
    }

In [ ]:
first = decide(claims[0], "openrouter/free")
first

### What the smaller prompt saves

Here's the same claim with Day 2's prompt: the whole policy wording, and the policy record and the claim as indented JSON. The two requests may go to different free models, which count tokens a little differently.

In [ ]:
def make_day2_messages(claim):
    """Day 2's prompt: the whole policy wording, and the policy record and the claim as indented JSON."""
    policy = policies[claim["policy_id"]]
    return [
        {"role": "system", "content": f"{INSTRUCTIONS}\n{policy_wording}\n"},
        {"role": "user", "content": f"# Policy record\n\n{json.dumps(policy, indent=2)}\n\n# Claim\n\n{json.dumps(claim, indent=2)}\n"},
    ]


day2 = ask("openrouter/free", make_day2_messages(claims[0]))

print(f"Day 2's prompt: {day2.usage.prompt_tokens:,} input tokens, answered by {day2.model}")
print(f"Today's prompt: {first['input_tokens']:,} input tokens, answered by {first['answered_by']}")

### Now running all 20 claims

This time we record whether the model gets the decision right, and whether it gets the decision and the payout right.

In [ ]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def score(answer, label):
    """Add the difficulty, and whether the decision, and the decision with the payout, match the handler's."""
    return {
        **answer,
        "difficulty": label["difficulty"],
        "decision_right": answer["decision"] == label["decision"],
        "right": is_right(answer, label),
    }


def report(results):
    """Print how a run did: the decisions right, the decisions right with the payout too, and the time."""
    decisions_right = sum(result["decision_right"] for result in results)
    right = sum(result["right"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{results[0]['model']}: {decisions_right} of {len(results)} decisions right, {right} with the payout too, in {seconds:.0f} seconds")

In [ ]:
def run(model, claims, labels):
    """Run a model on OpenRouter on the claims and compare against the labels."""
    results = []
    for claim in tqdm(claims, desc=model):
        answer = decide(claim, model)
        results.append(score(answer, labels[claim["claim_id"]]))
    report(results)
    return results

In [ ]:
free = run("openrouter/free", claims, labels)

These are the free models that answered, with the number of claims each one decided.

In [ ]:
pd.DataFrame(free)["answered_by"].value_counts()

## Part 3: Three models on your computer with Ollama

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Ollama</h2>
            <span style="color:#118AB2;">Ollama runs open-weight models on your own computer. Install it from <a href="https://ollama.com">ollama.com</a> and start it. It runs in the background and serves models on your computer. The <code>ollama</code> Python package talks to it.<br/><br/>
            Models are downloaded with a pull. Ollama loads a model into memory when you first use it and unloads it after a few minutes without use.
            </span>
        </td>
    </tr>
</table>

We run three models:

- `qwen3.5:4b-q4_K_M` is Qwen 3.5 with 4 billion parameters, a 3.3 GB download that fits a computer with 8 GB of memory
- `gemma4:e2b-it-qat` is Gemma 4 E2B, the instruction-tuned version trained to run at 4 bits, a 4.3 GB download that also fits 8 GB
- `gemma4:e4b-it-q4_K_M` is Gemma 4 E4B, a 6.6 GB download that needs a computer with 16 GB of memory. It's also the slowest of the three, so it's optional

These are llama.cpp (GGUF) builds. On a Mac, the plain `qwen3.5:4b` and `gemma4:e4b` tags can download as MLX builds, which fail here with "structured output is unavailable".

The pulls take a few minutes the first time.

In [ ]:
print(ollama.pull("qwen3.5:4b-q4_K_M").status)
print(ollama.pull("gemma4:e2b-it-qat").status)

In [ ]:
# Optional: a 6.6 GB download, and the slowest of the three
print(ollama.pull("gemma4:e4b-it-q4_K_M").status)

Ollama takes our Pydantic schema as `format` and holds the model's reply to it, so even a small model answers in valid JSON. `think=False` turns reasoning off to keep the small models quick. `num_ctx` sets the context window, which Ollama otherwise sets to 4,096 tokens whatever the model supports.

We also record how much memory Ollama is using for the model, which decides whether it fits on your computer.

In [ ]:
def memory_in_use(model):
    """The memory Ollama is using for a model it has loaded, in GB."""
    for running in ollama.ps().models:
        if running.model == model:
            return round(running.size / 1e9, 1)
    return None

In [ ]:
def decide_locally(claim, model):
    """Ask a model on this computer to decide one claim, through Ollama."""
    start = time.time()
    response = ollama.chat(
        model=model,
        messages=make_messages(claim),
        format=ClaimDecision.model_json_schema(),
        think=False,
        options={"num_ctx": 8192},
    )
    answer = ClaimDecision.model_validate_json(response.message.content)
    return {
        "model": model,
        "where": "This computer",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "input_tokens": response.prompt_eval_count,
        "memory": memory_in_use(model),
        "seconds": round(time.time() - start, 1),
    }

In [ ]:
def run_locally(model, claims, labels):
    """Run a model on this computer on the claims and compare against the labels."""
    results = []
    for claim in tqdm(claims, desc=model):
        answer = decide_locally(claim, model)
        results.append(score(answer, labels[claim["claim_id"]]))
    ollama.generate(model=model, keep_alive=0)  # unload the model, so the next one has the memory
    report(results)
    return results

In [ ]:
first_local = decide_locally(claims[0], "qwen3.5:4b-q4_K_M")
first_local

Now the 20 claims on each model. Each call runs on your own computer, so the time depends on your hardware.

In [ ]:
qwen = run_locally("qwen3.5:4b-q4_K_M", claims, labels)

In [ ]:
gemma_e2b = run_locally("gemma4:e2b-it-qat", claims, labels)

In [ ]:
# Optional: only if you pulled gemma4:e4b-it-q4_K_M
gemma_e4b = run_locally("gemma4:e4b-it-q4_K_M", claims, labels)

## Part 4: Accuracy, speed and memory side by side

One row for each model. Accuracy is the percentage of the 20 claims it got right, first on the decision alone and then on the decision and the payout together. Memory is what Ollama used on your computer, so it's blank for `openrouter/free`.

In [ ]:
# If you skipped Gemma 4 E4B, take gemma_e4b out of this list
experiments = [free, qwen, gemma_e2b, gemma_e4b]

rows = []
for results in experiments:
    count = len(results)
    rows.append({
        "model": results[0]["model"],
        "where": results[0]["where"],
        "decision right (%)": round(100 * sum(result["decision_right"] for result in results) / count),
        "decision and payout right (%)": round(100 * sum(result["right"] for result in results) / count),
        "seconds a claim": round(sum(result["seconds"] for result in results) / count, 1),
        "memory (GB)": results[0].get("memory"),
    })

summary = pd.DataFrame(rows)
summary

### Accuracy against speed

The best place to be is the top left: accurate and fast. The local times depend on your computer.

In [ ]:
fig = px.scatter(
    summary,
    x="seconds a claim",
    y="decision and payout right (%)",
    color="where",
    text="model",
    title="Accuracy against speed",
    height=600,
)
fig.update_traces(marker_size=14, textposition="top center")
fig.update_yaxes(range=[0, 100])
fig.show()

### Accuracy against memory

This one has the local models only. A computer with 8 GB of memory has room for a model that uses about 4 GB, since the operating system and your other apps need the rest.

In [ ]:
local = summary[summary["where"] == "This computer"]

fig = px.scatter(
    local,
    x="memory (GB)",
    y="decision and payout right (%)",
    text="model",
    title="Accuracy against memory",
    height=600,
)
fig.update_traces(marker_size=14, textposition="top center")
fig.update_yaxes(range=[0, 100])
fig.show()

### All three at once

The local models again, with memory, speed and accuracy. Drag the chart to turn it.

In [ ]:
fig = px.scatter_3d(
    local,
    x="memory (GB)",
    y="seconds a claim",
    z="decision and payout right (%)",
    text="model",
    title="Accuracy, speed and memory",
    height=700,
)
fig.show()

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">Weighing up accuracy, cost and latency for a business task is a crucial
            activity for an AI Solutions Architect and AI FDE.<br/><br/>
            There will also be situations where your data needs to stay on your customer's hardware.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Challenge: Try finding new ways to shorten the prompt that preserves the accuracy.
            Try finding a new way to get high accuracy results with as few tokens as possible.<br/><br/>
            Also try experimenting with more models on OpenRouter and locally.<br/><br/>
            Stretch challenge: Now go back to the product from yesterday and change it to use local models via Ollama!<br/><br/>
            Stretch stretch challenge: Investigate renting a GPU with RunPod, running a vLLM server, and running the product
            from that! Claude Code can give you instructions if you're interested..<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>